In [19]:
# Set up Dask cluster using Docker Python SDK
import docker

# Initialize Docker client
docker_client = docker.from_env()

# Create a Docker network
network_name = "dask-network"

try:
    docker_client.networks.get(network_name)
except docker.errors.NotFound:
    docker_client.networks.create(network_name, driver="bridge")

# Define paths for volume mounting
host_file_path = r'C:\Users\Adriano Matos\Documents\credentials\robust-raster-cefaedc5282c.json'  # Host machine path
container_file_path = '/credentials/robust-raster-cefaedc5282c.json'  # Path inside the container

# Volume mapping
volumes = {
    host_file_path: {'bind': container_file_path, 'mode': 'ro'},  # 'ro' means read-only
}

# Run Dask Scheduler
scheduler = docker_client.containers.run(
    "daskdev/dask-ee",
    command="dask-scheduler",
    name="dask-scheduler",
    network=network_name,
    detach=True,
    ports={'8786/tcp': 8786, '8787/tcp': 8787},
)

print(f"Dask Scheduler started with ID {scheduler.id}")

# Run Dask Workers
num_workers = 2  # Specify the number of workers you want
n_threads = 1
container_mem_limit = "16GB"
dask_mem_limit = "8GB"

worker_ids = []
for i in range(num_workers):
    worker = docker_client.containers.run(
        "daskdev/dask-ee",
        command=f"dask-worker dask-scheduler:8786 --nthreads {n_threads} --memory-limit {dask_mem_limit}",
        name=f"dask-worker-{i+1}",
        network=network_name,
        detach=True,
        mem_limit=container_mem_limit,
        volumes=volumes
)
    worker_ids.append(worker.id)
    print(f"Dask Worker {i+1} started with ID {worker.id}")

# Connect to the Dask Scheduler
#dask_client = Client('tcp://localhost:8786')

print("Connected to Dask Scheduler")

# Monitor the Dask Dashboard
print("Dask dashboard available at http://localhost:8787")

# The cluster is now set up and can be used within the script or interactively.

Dask Scheduler started with ID d6807b7f05aec09c00d77e8f7841115f318d2bcff752ccbdd58ef2ac708871b3
Dask Worker 1 started with ID 180b5cf3a009c0e5a61a3d09aa9ca2ab4ea91022724fcb03f49e01dc522e0f2c
Dask Worker 2 started with ID 92a1e37e565e1fcfe1a6341135d8af0b7802ceca3430bcf62c30f2e3be0f60c6
Connected to Dask Scheduler
Dask dashboard available at http://localhost:8787


In [1]:
from dask.distributed import Client
dask_client = Client('tcp://localhost:8786')

c:\Users\Adriano Matos\anaconda3\envs\robust_raster\lib\site-packages\distributed\client.py:1617: VersionMismatchWarning: Mismatched versions found

+---------+--------+-----------+---------+
| Package | Client | Scheduler | Workers |
+---------+--------+-----------+---------+
| toolz   | 0.12.1 | 0.12.0    | 0.12.0  |
+---------+--------+-----------+---------+
  warnings.warn(version_module.VersionMismatchWarning(msg[0]["warning"]))


In [21]:
import os
import json
import ee
from distributed import WorkerPlugin

class EEPlugin(WorkerPlugin):
    def __init__(self, json_key: str = None):
        self.json_key = json_key
    
    def setup(self, worker):
        self.worker = worker
        try:
            if self.json_key and os.path.exists(self.json_key):
                with open(self.json_key, 'r') as file:
                     data = json.load(file)
                credentials = ee.ServiceAccountCredentials(data["client_email"], self.json_key)
                ee.Initialize(credentials, opt_url='https://earthengine-highvolume.googleapis.com')
            else:
                ee.Initialize(opt_url='https://earthengine-highvolume.googleapis.com')
        except ee.EEException as e:
            if "Please authorize access to your Earth Engine account" in str(e):
                ee.Authenticate()
                ee.Initialize(opt_url='https://earthengine-highvolume.googleapis.com') 

    def teardown(self, worker):
        pass

    def transition(self, key, start, finish, **kwargs):
        pass

    def release_key(self, key, state, cause, reason, report):
        pass

ee_plugin = EEPlugin(container_file_path)


In [22]:
dask_client.register_plugin(ee_plugin)

{'tcp://172.20.0.3:43589': {'status': 'OK'},
 'tcp://172.20.0.4:37211': {'status': 'OK'}}

In [1]:
# Set up Dask cluster locally
import sys
import os

module_path = os.path.abspath(os.path.join(r'R:\SCRATCH\adrianom\code\big-raster-helper\src'))
if module_path not in sys.path:
    sys.path.append(module_path)
    
from initialize_dask import DaskHandler

handler = DaskHandler()
handler.create_local_cluster(n_workers=8, threads_per_worker=1, memory_limit="32GB")
#handler.initialize_earth_engine_dask_workers()
# If the crs is 4326 or anything geographic, I need a check to ensure that lon/lat are
# the dimension names!
# Maybe a method to close the cluster when done?

In [4]:
# EE credentials with a JSON key
import json
import ee
json_key = r'C:\Users\Adriano Matos\Documents\credentials\robust-raster-cefaedc5282c.json'
with open(json_key, 'r') as file:
    data = json.load(file)
credentials = ee.ServiceAccountCredentials(data["client_email"], json_key)
ee.Initialize(credentials = credentials, opt_url='https://earthengine-highvolume.googleapis.com')

In [2]:
initialize_dict = {
    'opt_url': 'https://earthengine-highvolume.googleapis.com',
    'project': 'calfuels'
}

In [ ]:
# Lazy operation! Does not load the raster into memory!!
da = xarray.open_dataset("https://github.com/mapbox/rasterio/raw/1.2.1/tests/data/RGB.byte.tif")

In [ ]:
###############################
### TESTING MY PACKAGE CODE ###
###############################

In [2]:
# Earth Engine xarray
import sys
import os
import ee
import os

ee.Initialize(opt_url='https://earthengine-highvolume.googleapis.com', project='robust-raster')

def prep_sr_l8(image):
    # Develop masks for unwanted pixels (fill, cloud, cloud shadow).
    qa_mask = image.select('QA_PIXEL').bitwiseAnd(int('11111', 2)).eq(0)
    saturation_mask = image.select('QA_RADSAT').eq(0)

    # Apply the scaling factors to the appropriate bands.
    def get_factor_img(factor_names):
        factor_list = image.toDictionary().select(factor_names).values()
        return ee.Image.constant(factor_list)
    
    scale_img = get_factor_img([
        'REFLECTANCE_MULT_BAND_.|TEMPERATURE_MULT_BAND_ST_B10'])
    offset_img = get_factor_img([
        'REFLECTANCE_ADD_BAND_.|TEMPERATURE_ADD_BAND_ST_B10'])
    scaled = image.select('SR_B.|ST_B10').multiply(scale_img).add(offset_img)

    # Replace original bands with scaled bands and apply masks.
    return image.addBands(scaled, None, True)\
        .updateMask(qa_mask).updateMask(saturation_mask)

module_path = os.path.abspath(os.path.join(r'C:\Users\Adriano Matos\Documents\Python Scripts\big-raster-helper\src'))
if module_path not in sys.path:
    sys.path.append(module_path)

from input_driver import EarthEngineData

CALIFORNIA = ee.FeatureCollection("projects/calfuels/assets/Boundaries/California")
LTBMU = ee.FeatureCollection("projects/calfuels/assets/Boundaries/LTBMU_remove_NV_remove_lake")

chunk_size  = {
            'time': 48,
            'X': 512,
            'Y': 256
}

parameters = {
            'collection': 'LANDSAT/LC08/C02/T1_L2',
            'bands': ['SR_B4', 'SR_B5'],
            'start_date': '2020-12-29',
            'end_date': '2020-12-31',
            'geometry': CALIFORNIA.geometry(),
            'crs': 'EPSG:3310',
            'scale': 100,
            'chunks': chunk_size,
            'map_function': prep_sr_l8
        }

landsat_xarray = EarthEngineData(parameters)

C:\Users\Adriano Matos\Documents\Python Scripts\big-raster-helper\src\input_driver.py:93: FutureWarning: The return type of `Dataset.dims` will be changed to return a set of dimension names in future, in order to be more consistent with `DataArray.dims`. To access a mapping from dimension names to lengths, please use `Dataset.sizes`.
  first_dimension = list(self._xarray_data.dims.keys())[0]


In [3]:
ds = landsat_xarray.dataset

In [4]:
print(ds)

<xarray.Dataset> Size: 2GB
Dimensions:  (time: 3, X: 9084, Y: 10578)
Coordinates:
  * time     (time) datetime64[ns] 24B 2020-12-29T18:57:32.281000 ... 2020-12...
  * X        (X) float64 73kB -4.216e+05 -4.215e+05 ... 4.866e+05 4.867e+05
  * Y        (Y) float64 85kB -5.992e+05 -5.991e+05 ... 4.584e+05 4.585e+05
Data variables:
    SR_B4    (time, X, Y) float32 1GB dask.array<chunksize=(3, 2097, 2097), meta=np.ndarray>
    SR_B5    (time, X, Y) float32 1GB dask.array<chunksize=(3, 2097, 2097), meta=np.ndarray>
Attributes: (12/26)
    date_range:             [1365638400000, 1654560000000]
    description:            <p>This dataset contains atmospherically correcte...
    keywords:               ['cfmask', 'cloud', 'fmask', 'global', 'l8sr', 'l...
    period:                 0
    product_tags:           ['global', 'sr', 'reflectance', 'l8sr', 'lst', 'c...
    provider:               USGS
    ...                     ...
    visualization_1_name:   Near Infrared (543)
    visualization_

In [ ]:
chunk_size  = {
            'time': 48,
            'X': 2048,
            'Y': 1024
}
ds_rechunked = ds.chunk(chunk_size)
print(ds_rechunked)

In [2]:
# Template xarray based on Earth Engine
import numpy as np
import pandas as pd
import xarray as xr

# Define the dimensions
time = pd.date_range("2020-12-29T18:57:32.281000", periods=3)
X = np.linspace(-421600, 486700, 9084)
Y = np.linspace(-599200, 458500, 10578)

# Create a data array with random data for each variable
data = np.random.rand(len(time), len(X), len(Y)).astype(np.float32)

# Create a dictionary of data variables
data_vars = {
    #'SR_B1': (['time', 'X', 'Y'], data),
    #'SR_B2': (['time', 'X', 'Y'], data),
    #'SR_B3': (['time', 'X', 'Y'], data),
    'SR_B4': (['time', 'X', 'Y'], data),
    'SR_B5': (['time', 'X', 'Y'], data),
    #'SR_B6': (['time', 'X', 'Y'], data),
    #'ST_EMSD': (['time', 'X', 'Y'], data),
    #'ST_QA': (['time', 'X', 'Y'], data),
    #'ST_TRAD': (['time', 'X', 'Y'], data),
    #'ST_URAD': (['time', 'X', 'Y'], data),
    #'QA_PIXEL': (['time', 'X', 'Y'], data),
    #'QA_RADSAT': (['time', 'X', 'Y'], data),
}

chunk_size = {'time': 3, 'X': 512, 'Y': 256}

# Create the dataset
ds = xr.Dataset(
    data_vars=data_vars,
    coords={'time': time, 'X': X, 'Y': Y},
    attrs={
        'date_range': '[1365638400000, 1654560000000]',
        'description': '<p>This dataset contains atmospherically corrected data.</p>',
        'keywords': ['cfmask', 'cloud', 'fmask', 'global', 'l8sr', 'landsat'],
        'period': 0,
        'visualization_2_max': 30000.0,
        'visualization_2_min': 0.0,
        'visualization_2_name': 'Shortwave Infrared (753)',
        'crs': 'EPSG:3310'
    }
).chunk(chunk_size)


In [ ]:
# Compute NDVI on an xarray
import dask.array as da
import xarray as xr
from dask.distributed import performance_report

def compute_ndvi(dataset):
    """
    Compute NDVI from an xarray Dataset using Dask.
    
    Parameters:
    dataset (xarray.Dataset): The input dataset containing NIR and Red bands.
    
    Returns:
    xarray.DataArray: NDVI values computed from the input dataset.
    """
    # Access the NIR (SR_B5) and Red (SR_B4) bands
    nir = dataset['SR_B5']
    red = dataset['SR_B4']
    
    # Ensure the arrays are dask arrays
    nir = nir.data if isinstance(nir, xr.DataArray) else nir
    red = red.data if isinstance(red, xr.DataArray) else red
    
    # Compute NDVI using Dask array operations
    ndvi = (nir - red) / (nir + red)
    
    # Convert the resulting dask array back to a DataArray
    ndvi_da = xr.DataArray(ndvi, dims=dataset['SR_B5'].dims, coords=dataset['SR_B5'].coords, name='NDVI')
    dataset['NDVI'] = ndvi_da
    
    return dataset

# Example usage with the dataset created earlier
ndvi = compute_ndvi(dataset)
ndvi.compute()
#with performance_report(filename='dask-report-nodf.html'):
#    ndvi.compute()

In [3]:
# Sample template dataset
import xarray as xr
import numpy as np
import dask.array as da

chunk_size = ds['SR_B4'].chunks

# Create a template with dask arrays using the same chunk sizes and attributes
template = xr.Dataset(
    {
        'SR_B4': (['time', 'X', 'Y'], da.empty((3, 9084, 10578), chunks=chunk_size, dtype=np.float32)),
        'SR_B5': (['time', 'X', 'Y'], da.empty((3, 9084, 10578), chunks=chunk_size, dtype=np.float32)),
        #'ndvi':  (['time', 'X', 'Y'], da.empty((3, 9084, 10578), chunks=chunk_size, dtype=np.float32))
    },
    coords={
        'time': ds.coords['time'],
        'X': ds.coords['X'],
        'Y': ds.coords['Y'],
    },
    attrs=ds.attrs  # Copy over the original dataset's attributes
)

In [3]:
# process_chunk_as_pandas function
import xarray as xr
import pandas as pd

def process_chunk_as_pandas(chunk):
    # Convert xarray chunk to pandas DataFrame
    df = chunk.to_dataframe().reset_index()
    
    # Perform your calculations
    df['ndvi'] = df['SR_B4'] + df['SR_B5']
    
    # Convert the pandas DataFrame back to an xarray structure
    df = df.set_index(list(chunk.dims))
    result = df.to_xarray()
    
    return result

In [26]:
# Create a function to generate the template based on the original dataset
def create_template(ds, process_func):
    # Extract a single chunk to determine the output structure
    one_chunk = ds.isel(time=slice(0, ds.chunks['time'][0]), 
                        X=slice(0, ds.chunks['X'][0]), 
                        Y=slice(0, ds.chunks['Y'][0]))
    
    # Apply the processing function to this chunk
    processed_chunk = process_func(one_chunk)
    
    # Create the template using a combination of original data variables and newly created ones
    template_vars = {}
    
    for var in processed_chunk.data_vars:
        if var in ds.data_vars:
            # Use the original dataset's shape and chunking for existing variables
            template_vars[var] = (processed_chunk[var].dims, 
                                  da.empty(ds[var].shape, 
                                           chunks=ds[var].chunks, 
                                           dtype=processed_chunk[var].dtype))
        else:
            # For new variables, define the shape and chunks manually based on the original chunking strategy
            new_var_shape = tuple(ds.dims[dim] for dim in processed_chunk[var].dims)
            new_var_chunks = tuple(ds.chunks[dim][0] for dim in processed_chunk[var].dims)
            template_vars[var] = (processed_chunk[var].dims, 
                                  da.empty(new_var_shape, 
                                           chunks=new_var_chunks, 
                                           dtype=processed_chunk[var].dtype))
    
    template = xr.Dataset(
        template_vars,
        coords={coord: ds.coords[coord] for coord in ds.coords},
        attrs=ds.attrs
    )
    
    return template

# Assuming `ds` is your chunked xarray object:
template = create_template(ds, process_chunk_as_pandas)

In [ ]:
# Assuming `your_chunked_xarray` is your chunked xarray object:
result = xr.map_blocks(process_chunk_as_pandas, ds, template=template)
result.compute()

In [4]:
##############################
### A SINGLE CHUNK OF DATA ###
##############################
dataset = landsat_xarray.dataset

# Select the first chunk (first time step, top-left corner) for SR_B5 and SR_B4
nir_chunk = dataset['SR_B5'].data.blocks[0, 0, 0]
red_chunk = dataset['SR_B4'].data.blocks[0, 0, 0]

# Define the NDVI computation function
def compute_ndvi(nir, red):
    return (nir - red) / (nir + red)

# Compute the NDVI for the selected chunk
ndvi_chunk = compute_ndvi(nir_chunk, red_chunk)

ndvi_chunk.compute()
# Visualize the task graph
#ndvi_chunk.visualize(filename='ndvi_chunk_task_graph_small.png')

In [ ]:
########################################################################
### Convert to data frame, run function, then convert back to xarray ###
########################################################################
def compute_ndvi(dataset):
    dask_df = dataset.to_dask_dataframe()
    # Ensure that the required columns are in the DataFrame
    if 'SR_B5' not in dask_df.columns or 'SR_B4' not in dask_df.columns:
        raise ValueError("DataFrame must contain 'SR_B5' and 'SR_B4' columns")

    # Compute NDVI
    dask_df['NDVI'] = (dask_df['SR_B5'] - dask_df['SR_B4']) / (dask_df['SR_B5'] + dask_df['SR_B4'])
    return dask_df
ndvi = compute_ndvi(dataset)

In [ ]:
########################################
### Compute NDVI on a Dask Dataframe ###
########################################

#dask_df = dataset.to_dask_dataframe().repartition(100)

def compute_ndvi(dask_df):
    # Ensure that the required columns are in the DataFrame
    if 'SR_B5' not in dask_df.columns or 'SR_B4' not in dask_df.columns:
        raise ValueError("DataFrame must contain 'SR_B5' and 'SR_B4' columns")

    # Compute NDVI
    dask_df['NDVI'] = (dask_df['SR_B5'] - dask_df['SR_B4']) / (dask_df['SR_B5'] + dask_df['SR_B4'])

    return dask_df

result = compute_ndvi(dask_df)
result.compute()

In [5]:
import xarray as xr
import pandas as pd

def process_chunk_as_pandas(df):
    # Perform your calculations
    df['ndvi'] = df['SR_B4'] + df['SR_B5']
    
    return df

In [6]:
import sys

module_path = os.path.abspath(os.path.join(r'R:\SCRATCH\adrianom\code\big-raster-helper\src'))
if module_path not in sys.path:
    sys.path.append(module_path)

from udf_tuner import UserDefinedFunction

user_defined_func = UserDefinedFunction()

# Apply the user-defined function
result = user_defined_func.apply_user_function(ds, process_chunk_as_pandas)

C:\Users\Adriano Matos\Documents\Python Scripts\big-raster-helper\src\udf_tuner.py:10: FutureWarning: The return type of `Dataset.dims` will be changed to return a set of dimension names in future, in order to be more consistent with `DataArray.dims`. To access a mapping from dimension names to lengths, please use `Dataset.sizes`.
  dim_names = list(ds.dims.keys())
C:\Users\Adriano Matos\Documents\Python Scripts\big-raster-helper\src\udf_tuner.py:31: FutureWarning: The return type of `Dataset.dims` will be changed to return a set of dimension names in future, in order to be more consistent with `DataArray.dims`. To access a mapping from dimension names to lengths, please use `Dataset.sizes`.
  new_var_shape = tuple(ds.dims[dim] for dim in processed_chunk[var].dims)


In [7]:
result.compute()

RuntimeError: Error during deserialization of the task graph. This frequently
occurs if the Scheduler and Client have different environments.
For more information, see
https://docs.dask.org/en/stable/deployment-considerations.html#consistent-software-environments


In [ ]:
def compute_ndvi(dask_df):
    # Ensure that the required columns are in the DataFrame
    if 'SR_B5' not in dask_df.columns or 'SR_B4' not in dask_df.columns:
        raise ValueError("DataFrame must contain 'SR_B5' and 'SR_B4' columns")

    # Compute NDVI
    dask_df['NDVI'] = (dask_df['SR_B5'] - dask_df['SR_B4']) / (dask_df['SR_B5'] + dask_df['SR_B4'])

    return dask_df

def apply_function(dataset, func, *args, **kwargs):
    """
    Apply a user-defined function to the Dask DataFrame.
        
    Parameters:
    - func: the user-defined function to apply.
    - args: positional arguments to pass to the function.
    - kwargs: keyword arguments to pass to the function.
        
    Returns:
    - result: the result of applying the function to the dataframe.
    """
    if not callable(func):
        raise ValueError("The provided function must be callable.")
    
    dask_df = dataset.to_dask_dataframe()
    result = dask_df.map_partitions(func, *args, **kwargs)
    result.compute()
    return result

    #report_name="report"
    #with performance_report(filename=f"{report_name}.html"):
    #    result.compute()
result = apply_function(dataset, compute_ndvi)